In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tableone import TableOne
import os
import datetime
from datetime import date
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score
from scipy.stats import ttest_ind
import re

# To Show All Columns in DataFrame or numpy arrays
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('max_colwidth', None)
# np.set_printoptions(threshold=np.inf)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)
pd.set_option('display.min_rows', 10)
pd.set_option('display.expand_frame_repr', True)


# To Get Rid of Warnings
import warnings
warnings.filterwarnings("ignore")

homepath = '/home/idies/workspace/'

In [2]:
df_outin_icd_lab = pd.read_csv('/projects/diabetic-eye-disease-nlp-hpc/df_outin_icd_lab.csv', header=0)

In [ ]:
#import updated lapse data 2025

In [9]:
df_update_lapses_250819 = pd.read_csv('/projects/diabetic-eye-disease-nlp-hpc/df_update_lapses_250819.csv', header=0)

# Data before the first lapse data to calculate CCI

In [12]:
df_outin_icd_lab_bef_first_lapse = df_outin_icd_lab.merge(df_update_lapses_250819[['e_mrn','enc_date_first_enc']].drop_duplicates(), how = 'inner', on ='e_mrn')

In [13]:
df_outin_icd_lab_bef_first_lapse['enc_date_first_enc'] = pd.to_datetime(df_outin_icd_lab_bef_first_lapse['enc_date_first_enc'])
df_outin_icd_lab_bef_first_lapse['appt_time'] = pd.to_datetime(df_outin_icd_lab_bef_first_lapse['appt_time'])

In [14]:
df_outin_icd_lab_bef_first_lapse_tmp = df_outin_icd_lab_bef_first_lapse.loc[df_outin_icd_lab_bef_first_lapse['appt_time'] <= df_outin_icd_lab_bef_first_lapse['enc_date_first_enc']]

# Caculate CCI

In [17]:
df_lapse_vision_outin_icd_lab_expl = df_outin_icd_lab_bef_first_lapse_tmp.copy()
#df_fsd_demo_ins_icd_adi_address_expl['current_icd10_list_rm'] = df_fsd_demo_ins_icd_adi_address_expl['current_icd10_list'].str.replace(r"(\d)\.", r"\1") #Remove dot from icd10
df_lapse_vision_outin_icd_lab_expl['current_icd10_list_rm'] = df_lapse_vision_outin_icd_lab_expl['current_icd10_list'].str.replace('.', '') #Remove dot from icd10
n_cols = df_lapse_vision_outin_icd_lab_expl['current_icd10_list_rm'].str.count(',').value_counts().shape[0]
df_lapse_vision_outin_icd_lab_expl_split = df_lapse_vision_outin_icd_lab_expl['current_icd10_list_rm'].str.rsplit(', ', n = n_cols, expand=True).add_prefix('icd10_').join(df_lapse_vision_outin_icd_lab_expl)

In [20]:
#Calculate Chalson index in STATA
df_lapse_vision_outin_icd_lab_expl_split.to_csv('/home/dtran30/persistent/060925 CCI DCSI/df_lapse_cci_input_stata.csv', index=False) 

# Import from SATAT result

In [21]:
df_lapse_cci = pd.read_csv('/home/dtran30/persistent/060925 CCI DCSI/df_lapse_cci_output_stata.csv')

In [157]:
#pd.set_option('display.max_columns', 100)

In [23]:
# Chose the hightest charindex
df_lapse_vision_cci_cci_max = df_lapse_cci[['e_mrn','charlindex']].sort_values('charlindex', ascending = False).groupby(by=['e_mrn'], as_index=False).first()
df_lapse_vision_cci_cci_max = df_lapse_vision_cci_cci_max.rename(columns={'charlindex': 'CCI' })
df_lapse_vision_outin_icd_lab_cci = df_outin_icd_lab_bef_first_lapse_tmp.merge(df_lapse_vision_cci_cci_max, how = 'left', on = 'e_mrn')

# Caculate DCSI

In [24]:
#https://regex101.com/
#https://regexr.com/39df2

#dcsi_app1_reg1 = r'(E(08|09|10|11|13)\.(?!34|35)\d+)|(H35\.[0|6|8]\d+)|(H35\.35\d+)|(H35\.9)'
dcsi_app1_reg1 = r'(E(08|09|10|11|13)\.(3[01236879])\d*)|(H35\.[0|6|8]\d+)|(H35\.35\d+)|(H35\.9$)'
dcsi_app1_reg2 = r'(H33\.\d+)|(E(08|09|10|11|13)\.(34|35)\d+)|(H54\.\d+)|(H43\.1\d+)'

#dcsi_app2_reg1 = r'(E(08|09|10|11|13)\.(21|22|29))|(N(00|03|04|05)\.\d+)|(N18\.(1|2|3|9))'
dcsi_app2_reg1 = r'(E(08|09|10|11|13)\.(21|22|29)$)|(N(00|03|04|05)\.\d+)|(N18\.(1|2|3|9)$)'
dcsi_app2_reg2 = r'(N18\.(4|5|6|9)$)|(N19$)'

#dcsi_app3_reg1 = r'(E(08|09|10|11|13)\.4\d+)|(G90\.(01|09|0|8|9|))|(G99\.0)|(G5(6|7)\.\d+)|(G60\.9)|(G73\.3)|(H49\.\d+)|(I95\.1)|(K31\.84)|(K59\.1)|(N31\.9)|(M14\.6\d+)|(S04\.\d+)'
dcsi_app3_reg1 = r'(E(08|09|10|11|13)\.4\d+)|(G90\.(01|09|0|8|9)$)|(G99\.0)|(G5(6|7)\.\d+)|(G60\.9$)|(G73\.3$)|(H49\.\d+)|(I95\.1$)|(K31\.84$)|(K59\.1$)|(N31\.9$)|(M14\.6\d+)|(S04\.\d+)'

dcsi_app4_reg1 = r'(G45\.\d+)'
#dcsi_app4_reg2 = r'(I6(1|3|5|6)\.\d+)|(I67\.81)'
dcsi_app4_reg2 = r'(I6(1|3|5|6)\.\d+)|(I67\.81$)'

#dcsi_app5_reg1 = r'(I24\.\d+)|(I20\.\d+)|(I25\.(?!2)\d+)|(I70\.(?!25|26\d+)\d+)'
dcsi_app5_reg1 = r'(I24\.\d+)|(I20\.\d+)|((I25\.)(?!2$)(\d+))|(I70\.(?!25|26\d+)\d+)'
#dcsi_app5_reg2 = r'(I2[1-3]\.\d+)|(I25\.2)|(I4[6-9]\.\d+)|(I50\.\d+)|(I70\.25)|(I70\.26\d+)|(I71\.\d+)'
dcsi_app5_reg2 = r'(I2[1-3]\.\d+)|(I25\.2$)|(I4[6-9]\.\d+)|(I50\.\d+)|(I70\.25$)|(I70\.26\d+)|(I71\.\d+)'

#dcsi_app6_reg1 = r'(E(08|09|10|11|13)\.(51|59|621))|(I72\.4)|(I70\.21\d+)|(I73\.(89|9))|(S91\.3\d+)'
dcsi_app6_reg1 = r'(E(08|09|10|11|13)\.(51|59|621)$)|(I72\.4$)|(I70\.21\d+)|(I73\.(89|9)$)|(S91\.3\d+)'
#dcsi_app6_reg2 = r'(A48\.0)|(I74\.3)|(L97\.\d+)|(E(08|09|10|11|13)\.52)|(I96)'
dcsi_app6_reg2 = r'(A48\.0$)|(I74\.3$)|(L97\.\d+)|(E(08|09|10|11|13)\.52$)|(I96$)'

#dcsi_app7_reg1 = r'(E(08|09|10|11|13)\.(00|10|649))'
dcsi_app7_reg1 = r'(E(08|09|10|11|13)\.(00|10|649)$)'
#dcsi_app7_reg2 = r'(E(08|09|10|11|13)\.(01|11|641))'
dcsi_app7_reg2 = r'(E(08|09|10|11|13)\.(01|11|641)$)'

In [25]:
#df[df["A"].str.contains(fr'\b{variable}\b', regex=True, case=False)]
#df_fsd_demo_ins_icd_adi_address_cci['current_icd10_list'].str.contains('Z13.5')[0:5]
#df_fsd_demo_ins_icd_adi_address_cci["current_icd10_list"].str.contains(dcsi_app1_reg1, regex=True, case=False).size
df_lapse_vision_dcsi_11 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app1_reg1, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_11 = df_lapse_vision_dcsi_11.rename(columns={'current_icd10_list':'dcsi11'})

df_lapse_vision_dcsi_12 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app1_reg2, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_12 = df_lapse_vision_dcsi_12.rename(columns={'current_icd10_list':'dcsi12'})

df_lapse_vision_dcsi_21 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app2_reg1, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_21 = df_lapse_vision_dcsi_21.rename(columns={'current_icd10_list':'dcsi21'})

df_lapse_vision_dcsi_22 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app2_reg2, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_22 = df_lapse_vision_dcsi_22.rename(columns={'current_icd10_list':'dcsi22'})

df_lapse_vision_dcsi_31 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app3_reg1, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_31 = df_lapse_vision_dcsi_31.rename(columns={'current_icd10_list':'dcsi31'})

df_lapse_vision_dcsi_41 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app4_reg1, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_41 = df_lapse_vision_dcsi_41.rename(columns={'current_icd10_list':'dcsi41'})

df_lapse_vision_dcsi_42 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app4_reg2, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_42 = df_lapse_vision_dcsi_42.rename(columns={'current_icd10_list':'dcsi42'})

df_lapse_vision_dcsi_51 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app5_reg1, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_51 = df_lapse_vision_dcsi_51.rename(columns={'current_icd10_list':'dcsi51'})

df_lapse_vision_dcsi_52 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app5_reg2, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_52 = df_lapse_vision_dcsi_52.rename(columns={'current_icd10_list':'dcsi52'})

df_lapse_vision_dcsi_61 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app6_reg1, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_61 = df_lapse_vision_dcsi_61.rename(columns={'current_icd10_list':'dcsi61'})

df_lapse_vision_dcsi_62 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app6_reg2, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_62 = df_lapse_vision_dcsi_62.rename(columns={'current_icd10_list':'dcsi62'})

df_lapse_vision_dcsi_71 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app7_reg1, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_71 = df_lapse_vision_dcsi_71.rename(columns={'current_icd10_list':'dcsi71'})

df_lapse_vision_dcsi_72 = df_lapse_vision_outin_icd_lab_cci["current_icd10_list"].str.contains(dcsi_app7_reg2, regex=True, case=False).to_frame()
df_lapse_vision_dcsi_72 = df_lapse_vision_dcsi_72.rename(columns={'current_icd10_list':'dcsi72'})

In [26]:
df_dsci_add = df_lapse_vision_outin_icd_lab_cci.merge(df_lapse_vision_dcsi_11,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_12,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_21,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_22,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_31,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_41,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_42,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_51,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_52,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_61,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_62,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_71,left_index=True, right_index=True)
df_dsci_add = df_dsci_add.merge(df_lapse_vision_dcsi_72,left_index=True, right_index=True)

In [27]:
df_dsci_add_drop = df_dsci_add.copy()

cols1 = ['dcsi11','dcsi21','dcsi31','dcsi41','dcsi51','dcsi61','dcsi71']
df_dsci_add_drop[cols1] = df_dsci_add_drop[cols1].replace({True:1, False:0})

cols2 = ['dcsi12','dcsi22','dcsi42','dcsi52','dcsi62','dcsi72']
df_dsci_add_drop[cols2] = df_dsci_add_drop[cols2].replace({True:2, False:0})

df_dsci_add_drop["dcsi_a1"] = df_dsci_add_drop[["dcsi11", "dcsi12"]].max(axis=1)
df_dsci_add_drop["dcsi_a2"] = df_dsci_add_drop[["dcsi21", "DCSI_lab_1", "dcsi22", "DCSI_lab_2"]].max(axis=1)
df_dsci_add_drop["dcsi_a3"] = df_dsci_add_drop[["dcsi31"]]
df_dsci_add_drop["dcsi_a4"] = df_dsci_add_drop[["dcsi41", "dcsi42"]].max(axis=1)
df_dsci_add_drop["dcsi_a5"] = df_dsci_add_drop[["dcsi51", "dcsi52"]].max(axis=1)
df_dsci_add_drop["dcsi_a6"] = df_dsci_add_drop[["dcsi61", "dcsi62"]].max(axis=1)
df_dsci_add_drop["dcsi_a7"] = df_dsci_add_drop[["dcsi71", "dcsi72"]].max(axis=1)

df_dsci_add_drop = df_dsci_add_drop.drop(columns=cols1+cols2)
#df_dsci_add_drop = df_dsci_add_drop.drop(cols, axis=1, inplace=True)

In [28]:
#df[cols_to_sum].sum(axis=1)
df_dsci_add_drop_sum = df_dsci_add_drop.copy()
cols_to_sum = ['dcsi_a1','dcsi_a2','dcsi_a3','dcsi_a4','dcsi_a5','dcsi_a6','dcsi_a7']
df_dsci_add_drop_sum['dcsi'] = df_dsci_add_drop_sum[cols_to_sum].sum(axis=1,min_count=1)

df_lapse_vision_dcsi = df_dsci_add_drop_sum.drop(columns=cols_to_sum)

In [29]:
# Chose the hightest charindex
df_lapse_vision_dcsi_max = df_lapse_vision_dcsi[['e_mrn','dcsi']].sort_values('dcsi', ascending = False).groupby(by=['e_mrn'], as_index=False).first()
df_lapse_vision_dcsi_max = df_lapse_vision_dcsi_max.rename(columns={'dcsi': 'DCSI' })
df_lapse_vision_dcsi_cci = df_lapse_vision_outin_icd_lab_cci.merge(df_lapse_vision_dcsi_max, how = 'left', on = 'e_mrn')
df_CCI_DCSI = df_lapse_vision_dcsi_cci[['e_mrn','CCI', 'DCSI']].drop_duplicates()

In [32]:
#Calculate Chalson index in STATA
df_CCI_DCSI.to_csv('/home/dtran30/persistent/060925 CCI DCSI/df_cci_dcsi_090625.csv', index=False)